## PII+Image Redactor Example Notebook


**Author**: Shahrokh Daijavad,
**email**:shahrokh@us.ibm.com

### Summary

In the [PII recipe](../PII/Run_your_first_PII_redactor_transform.ipynb), it is shown how the basic PII (Personally Identifiable Information) Redactor is used to identify and redact sensitive information in text data, such as  
Names, Email addresses, Phone numbers, Addresses, and Financial details (e.g., credit card numbers and crypto addresses). Here we want to show one of the Multimedia tranforms in DPK that will be used to blur the face of an image in a document, as an additional step in applying PII redaction beyond text. 

 **Workflow Overview**

- **Extracting and Converting Text and Image:** The content of a hypothetical invoice, originally in PDF format, is processed using the docling2parquet transform to extract both the text and image and convert them into a structured Parquet file, enabling easier handling and downstream processing by other DPK transforms. 

- **Redacting Sensitive Text Information:** The generated Parquet file serves as the input for the dpk_pii_redactor transform. This step scans the invoice data for personally identifiable information (PII) and applies masking techniques to redact any sensitive content, ensuring data privacy and compliance.

- **Redacting Image using a face-blurring technique:** The generated output Parquet file from the previous stage serves as the input for the dpk_mm/people transform. This step scans the input file, detects the face in the image and blurrs the face for additional data privacy. 

- **Creating the Final Output:** After the redaction processes for both text and image, a new output Parquet file is generated in **output-redacted** folder, containing the same structured data as the original but with all sensitive details securely masked to prevent unauthorized access or exposure.


## How to run this notebook

If you have python 3.11 or higher on your machine, you can also download the notebook and run it locally using a local python environment setup as follows:

```
python -m venv venv
source venv/bin/activate
pip install jupyterlab
jupyter lab PII_Image_redactor.ipynb
```

For more advanced setup, please see setup [guide](https://github.com/data-prep-kit/data-prep-kit/blob/dev/doc/quick-start/quick-start.md).


### Pre-req: Install data-prep-kit toolkit

In [ ]:

%pip install "data-prep-toolkit-transforms==1.1.7.dev3"
%pip install pandas 
import pandas as pd
import os

## Step 1: Configuration

### Download Data and set input and output directories
#### We will place the downloaded files in the `input_dir`. For our use case, we have used Invoice data file that will undergo processing. The output for each transform run will be generated in separate directories, with directory names following the format `files_<transform_name>`, making it easy to verify the respective transform outputs. This concludes the setup section.

In [ ]:
import urllib.request
import shutil
shutil.os.makedirs("tmp/input", exist_ok=True)
#urllib.request.urlretrieve("https://raw.githubusercontent.com/data-prep-kit/data-prep-kit/dev/recipes/input-data/PII-image/Invoiceplusimage.pdf", "tmp/input/Invoiceplusimage.pdf")

input_dir = "tmp/input"
output_dir = "output"
output_docling2pq_dir = os.path.join (output_dir, 'files_docling2parquet')
output_piiredactor_dir = os.path.join (output_dir, 'files_piiredacted')
output_people_dir = os.path.join (output_dir, 'files_people')

## Display the input PDF file

In [ ]:
from IPython.display import IFrame
IFrame(src=f"{input_dir}/Invoiceplusimage.pdf", width=600, height=800)

## Step 2: Invoke Docling2Parquet transform to proces pdf files

In [ ]:
from dpk_docling2parquet import Docling2Parquet
from data_processing.utils import GB
from dpk_docling2parquet.transform import docling2parquet_contents_types

In [ ]:

%%time

from dpk_docling2parquet import Docling2Parquet
from data_processing.utils import GB
from dpk_docling2parquet import docling2parquet_contents_types

STAGE = 1
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{input_dir}' --> output='{output_docling2pq_dir}'\n", flush=True)

Docling2Parquet(input_folder= input_dir,
               output_folder= output_docling2pq_dir,
               data_files_to_use=['.pdf'],
               docling2parquet_contents_type=docling2parquet_contents_types.MARKDOWN,
               docling2parquet_generate_picture_images=True,
               docling2parquet_pipeline="vlm").transform()

## Step 3: Invoke PII Redactor redaction transform

In [ ]:
%%time

from dpk_pii_redactor import PIIRedactor

STAGE = 2
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_docling2pq_dir}' --> output='{output_piiredactor_dir}'\n", flush=True)
PIIRedactor(input_folder=output_docling2pq_dir,
            output_folder= output_piiredactor_dir,
            pii_redactor_entities = ["PERSON", "EMAIL_ADDRESS","ORGANIZATION","PHONE_NUMBER", "LOCATION","CRYPTO"],
            pii_redactor_operator = "replace",
            pii_redactor_transformed_contents = "title").transform()

## Step 4: Display Output in a Readable Format with masked PII information

In [ ]:
data = pd.read_parquet('output/files_piiredacted/Invoiceplusimage.parquet')
print(data["title"][0])
print(data["detected_pii"][0])

## Step 5: Invoke People transform to blur the image from the Invoice file 

In [ ]:
%pip install "data-prep-toolkit-transforms[people]==1.1.7.dev3"

In [ ]:
%%time

from dpk_people import People

STAGE = 3
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_piiredactor_dir}' --> output='{output_people_dir}'\n", flush=True)

People(input_folder=output_piiredactor_dir,
    output_folder=output_people_dir,
).transform()

In [ ]:
import glob
glob.glob("output_people_dir/*")

In [ ]:
import pyarrow.parquet as pq
import pandas as pd

# Read the Parquet file into an Arrow Table
df = pq.read_table('output/files_people/Invoiceplusimage.parquet').to_pandas()
df.head()

In [ ]:
import io
from IPython.display import display
import PIL.Image as Image
    
image = Image.open(io.BytesIO(df.iloc[0]['blurred_images'][0]))
display(image)


### This notebook effectively demonstrates how to seamlessly apply redaction for PII entities.